# util — Trackastra→tracksdata Phase-2 pivot: OFFLINE ILP kill-criterion

We are reproducing the paradigm of the public **0.897** notebook (`references/biohub-cell-tracking-blend-preprocessings`,
pilkwang's learned-graph): **learned temporal-UNet detector + learned edge scorer + GLOBAL ILP linking (SCIP) + native
divisions**, all on royerlab's **`tracksdata`** graph library. Their submission reruns with internet OFF, so an offline
ILP is already proven feasible in principle — this notebook is our own **kill-criterion** to confirm we can package and
run it, before investing in the model.

**The single hard question:** *can `tracksdata`'s `ILPSolver` (SCIP via `pyscipopt`/`ilpy`) install and solve a tracking
problem on Kaggle with the network fully blocked?* If yes, the whole learned-graph paradigm is offline-viable.

| # | gate question | how it's tested |
|---|---|---|
| A | offline install works | `pip download` wheels (internet ON) → `pip install --no-index` (internet OFF), rc 0 + imports OK |
| B | SCIP ILP solves, no license, sockets blocked | build a small tracking graph → `ILPSolver.solve()` with `socket` hard-blocked |
| C | lineage supports divisions | check the solved graph can carry out-degree ≥ 2 nodes (division cost is a first-class `ILPSolver` param) |

**How to run (two phases, like the Trackastra gate):**
1. `PHASE='download'` (env `TD_PHASE=download`), **internet ON**: builds `/kaggle/working/wheels`, strips numpy/scipy
   (protect the C-ABI, per the Trackastra lesson), then installs + **probes the exact tracksdata API**. Save
   `/kaggle/working/wheels` as a Kaggle dataset **`tracksdata-offline`**.
2. `PHASE='offline_test'` (env `TD_PHASE=offline_test`), **internet OFF**, attach `tracksdata-offline`: installs from the
   wheels with sockets hard-blocked and runs gates A/B/C in a fresh subprocess.

⚠️ Lessons carried over from the Trackastra gate: run heavy work in a **fresh subprocess** (numpy py/C ABI), **don't
pin/ship numpy or scipy** offline (subprocess + stripped wheels isolate the Kaggle base), install with `--no-index`.


In [1]:
import os, sys, subprocess, glob, shutil
from pathlib import Path

PHASE = os.environ.get('TD_PHASE', 'offline_test')   # 'download' (internet ON) | 'offline_test' (internet OFF)

WORKING = Path('/kaggle/working')
WHEELS_DIR = WORKING / 'wheels'
OFFLINE_WHEELS_CANDIDATES = [
    Path('/kaggle/input/tracksdata-offline/wheels'),
    Path('/kaggle/input/tracksdata-offline'),
    WHEELS_DIR,
]

# Curated from the 0.897 reference (its PACKAGE_SPECS) so the offline install mirrors a proven-offline submission.
# tracksdata + the SCIP ILP backend (pyscipopt / ilpy) are the core; the rest are its transitive deps.
PIP_SPECS = [
    'tracksdata', 'pyscipopt', 'ilpy>=0.5.1',
    'zarr>=3.0.10,<4', 'geff>=1.1.3.1.1', 'geff-spec<1.2',
    'polars>=1.36', 'blosc2', 'dask', 'imagecodecs', 'scikit-image>=0.24',
    'pyarrow', 'rustworkx>=0.17.1', 'sqlalchemy>=2', 'numcodecs>=0.13,<0.16',
    'donfig>=0.8', 'google-crc32c>=1.5', 'bidict>=0.23.1', 'psygnal>=0.14',
    'rich', 'networkx>=3.2.1', 'pydantic>=2.11', 'pydantic-core',
    'annotated-types', 'typing-extensions>=4.13', 'typing-inspection',
    'markdown-it-py', 'pygments', 'click', 'cloudpickle', 'fsspec',
    'partd', 'locket', 'toolz', 'pyyaml', 'ndindex', 'msgpack',
    'numexpr', 'deprecated', 'wrapt',
]
# numpy/scipy intentionally NOT shipped offline (protect the Kaggle C-ABI; the subprocess + strip isolate the base).
STRIP_PREFIXES = ('numpy-', 'scipy-')
print('PHASE =', PHASE)


PHASE = offline_test


## Phase 1 (internet ON) — build the offline wheelhouse

`pip download` the whole dependency closure into `/kaggle/working/wheels`, then strip numpy/scipy wheels so the offline
install can never upgrade them. Save the result as the Kaggle dataset `tracksdata-offline`.


In [2]:
if PHASE != 'download':
    print('skip — set TD_PHASE=download (internet ON) to build the wheelhouse')
else:
    WHEELS_DIR.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, '-m', 'pip', 'download', '--dest', str(WHEELS_DIR)] + PIP_SPECS
    print('pip download ->', WHEELS_DIR)
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-2500:]); print('--- stderr tail ---'); print(r.stderr[-1500:]); print('returncode', r.returncode)
    removed = []
    for w in list(WHEELS_DIR.glob('*')):
        if w.name.lower().startswith(STRIP_PREFIXES):
            removed.append(w.name); w.unlink()
    print('stripped ABI wheels:', removed)
    wheels = sorted(WHEELS_DIR.glob('*'))
    print('wheelhouse:', len(wheels), 'files,', round(sum(w.stat().st_size for w in wheels)/1e6, 1), 'MB')
    print('NEXT: save /kaggle/working/wheels as a Kaggle dataset named tracksdata-offline, then run PHASE=offline_test.')


skip — set TD_PHASE=download (internet ON) to build the wheelhouse


## Phase 1 — probe the exact tracksdata API (internet ON)

Installs from the just-built wheelhouse and prints the real signatures (`ILPSolver`, graph `add_node`/`add_edge`,
`DEFAULT_ATTR_KEYS`, solution read-back). Use this output to finalize the division read-back in the offline gate below.


In [3]:
PROBE = r'''
import inspect, os
os.environ.setdefault('POLARS_PREFER_PKG', '32')
import tracksdata as td
print('tracksdata', getattr(td, '__version__', '?'))
from tracksdata import solvers, graph as tdgraph, nodes as tdnodes, edges as tdedges
print('ILPSolver.__init__:', inspect.signature(solvers.ILPSolver.__init__))
print('NearestNeighborsSolver.__init__:', inspect.signature(solvers.NearestNeighborsSolver.__init__))
g = tdgraph.InMemoryGraph()
print('InMemoryGraph public methods:', [m for m in dir(g) if not m.startswith('_')])
for m in ('add_node','add_edge','add_node_attr_key','add_edge_attr_key','node_ids','nodes',
          'edge_attrs','node_attrs','out_edges','successors','num_nodes','num_edges'):
    if hasattr(g, m):
        try: print('  sig', m, inspect.signature(getattr(g, m)))
        except Exception as e: print('  sig', m, '<'+repr(e)+'>')
try:
    from tracksdata.constants import DEFAULT_ATTR_KEYS
    print('DEFAULT_ATTR_KEYS:', {k: getattr(DEFAULT_ATTR_KEYS, k) for k in dir(DEFAULT_ATTR_KEYS) if not k.startswith('_')})
except Exception as e:
    print('DEFAULT_ATTR_KEYS import:', repr(e))
print('RandomNodes.__init__:', inspect.signature(tdnodes.RandomNodes.__init__))
print('DistanceEdges.__init__:', inspect.signature(tdedges.DistanceEdges.__init__))
if hasattr(tdgraph, 'GraphView'):
    print('GraphView public methods:', [m for m in dir(tdgraph.GraphView) if not m.startswith('_')])
'''

if PHASE != 'download':
    print('skip — the probe runs in the download phase (internet ON)')
else:
    wheel_files = sorted(str(w) for w in WHEELS_DIR.glob('*.whl'))
    inst = [sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps'] + wheel_files
    print('installing from wheelhouse ...')
    ri = subprocess.run(inst, capture_output=True, text=True)
    print(ri.stdout[-1500:]); print('--- stderr tail ---'); print(ri.stderr[-1500:]); print('install rc', ri.returncode)
    Path('/tmp/td_probe.py').write_text(PROBE)
    rp = subprocess.run([sys.executable, '/tmp/td_probe.py'], capture_output=True, text=True)
    print('===== PROBE OUTPUT ====='); print(rp.stdout); print('--- probe stderr ---'); print(rp.stderr[-1500:])


skip — the probe runs in the download phase (internet ON)


## Phase 2 (internet OFF) — install from wheels + solve the ILP with the network hard-blocked

Attach the `tracksdata-offline` dataset, turn internet OFF, set `TD_PHASE=offline_test`. Installs from the wheels, then
runs the gate in a fresh subprocess that hard-blocks all sockets before importing anything. **Gate A+B (install + SCIP
ILP solves offline) is the hard kill-criterion; Gate C (divisions) is a capability check.**


In [4]:
GATE = r'''
import sys, socket
from collections import Counter

# ---- hard-block the network (prove no internet is needed) ----
class _NoNet(socket.socket):
    def connect(self, *a, **k): raise OSError('NETWORK HARD-BLOCKED (offline gate)')
    def connect_ex(self, *a, **k): raise OSError('NETWORK HARD-BLOCKED (offline gate)')
def _blocked(*a, **k): raise OSError('NETWORK HARD-BLOCKED (offline gate)')
socket.socket = _NoNet
socket.create_connection = _blocked
socket.getaddrinfo = _blocked
print('[net] sockets hard-blocked')

res = {}
import os
os.environ.setdefault('POLARS_PREFER_PKG', '32')
import numpy as np
import tracksdata as td
try:
    import pyscipopt; scip = getattr(pyscipopt, '__version__', '?')
except Exception as e:
    scip = 'FAIL:'+repr(e)
try:
    import ilpy; ilpyv = getattr(ilpy, '__version__', '?')
except Exception as e:
    ilpyv = 'FAIL:'+repr(e)
print('tracksdata', getattr(td,'__version__','?'), '| numpy', np.__version__, '| pyscipopt', scip, '| ilpy', ilpyv)
res['imports'] = ('FAIL' not in str(scip)) and ('FAIL' not in str(ilpyv))

# ---- Gate B: ILPSolver solves a tracking graph with the network blocked ----
g = td.graph.InMemoryGraph()
td.nodes.RandomNodes(n_time_points=6, n_nodes_per_tp=(8, 12), n_dim=2).add_nodes(g)
td.edges.DistanceEdges(distance_threshold=0.35, n_neighbors=3).add_edges(g)
print('candidate graph:', g.num_nodes(), 'nodes /', g.num_edges(), 'edges')
solver = td.solvers.ILPSolver(edge_weight=-1.0, appearance_weight=0.1, disappearance_weight=0.1, division_weight=0.0)  # edge reward mirrors the 0.897 ref (candidate edges are pre-gated short by DistanceEdges)
sol = solver.solve(g)
assert sol is not None, 'ILPSolver returned None'
print('SOLUTION:', sol.num_nodes(), 'nodes /', sol.num_edges(), 'edges')
res['ilp_solved_offline'] = sol.num_edges() > 0

# ---- Gate C: lineage supports divisions (out-degree >= 2 somewhere) ----
# division_weight=0.0 above lets the solver split freely, so out-degree 2 should appear where candidates allow it.
try:
    degs = None
    if hasattr(sol, 'edge_attrs'):
        df = sol.edge_attrs()
        srccol = None
        for key in ('source_id', 'source', 'edge_source'):
            try:
                srccol = df[key]; break
            except Exception:
                pass
        if srccol is not None:
            degs = Counter(list(srccol))
    if degs is None and hasattr(sol, 'node_ids') and hasattr(sol, 'out_edges'):
        degs = {n: len(list(sol.out_edges(n))) for n in sol.node_ids()}
    if degs is not None and len(degs):
        maxdeg = max(degs.values())
        ndiv = sum(1 for v in degs.values() if v >= 2)
        print('max out-degree in solution:', maxdeg, '| division-like nodes:', ndiv)
        res['division_capable'] = bool(maxdeg >= 2)
    else:
        print('could not read solution edges — adjust the read-back API using the Phase-1 probe output')
        res['division_capable'] = 'UNKNOWN (read-back API TBD)'
except Exception as e:
    print('division check error:', repr(e))
    res['division_capable'] = 'ERROR: '+repr(e)

print('==== GATE SUMMARY ====')
for k, v in res.items():
    print('  ', k, ':', v)
hard = bool(res.get('imports')) and bool(res.get('ilp_solved_offline'))
print('HARD KILL-CRITERION (offline SCIP ILP install + solve):', 'PASS' if hard else 'FAIL')
'''

def _find_wheels_dir():
    # 1) explicit candidates
    for cand in OFFLINE_WHEELS_CANDIDATES:
        if cand.exists() and list(cand.glob('*.whl')):
            return cand
    # 2) auto-discover under /kaggle/input regardless of the dataset name (hyphen/underscore/etc.)
    inp = Path('/kaggle/input')
    if inp.exists():
        hits = list(inp.glob('**/tracksdata-*.whl')) + list(inp.glob('**/tracksdata_*.whl'))
        if hits:
            return hits[0].parent
        from collections import Counter
        dirs = Counter(w.parent for w in inp.glob('**/*.whl'))
        if dirs:
            return dirs.most_common(1)[0][0]
    return None

wdir = _find_wheels_dir()
print('offline wheelhouse:', wdir)
if wdir is not None:
    print('  wheels found:', len(list(wdir.glob('*.whl'))))

if PHASE != 'offline_test':
    print('skip — set TD_PHASE=offline_test with internet OFF and the tracksdata-offline dataset attached')
else:
    assert wdir is not None, 'attach the tracksdata-offline dataset (wheels) first'
    wheel_files = sorted(str(w) for w in wdir.glob('*.whl'))
    inst = [sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps'] + wheel_files
    print('installing from wheels (no network) ...')
    ri = subprocess.run(inst, capture_output=True, text=True)
    print(ri.stdout[-1500:]); print('--- stderr tail ---'); print(ri.stderr[-1500:]); print('install rc', ri.returncode)
    Path('/tmp/td_gate.py').write_text(GATE)
    rg = subprocess.run([sys.executable, '/tmp/td_gate.py'], capture_output=True, text=True)
    print('===== GATE OUTPUT ====='); print(rg.stdout); print('--- gate stderr tail ---'); print(rg.stderr[-2000:]); print('gate rc', rg.returncode)


offline wheelhouse: /kaggle/input/datasets/lingxd/tracksdata-offline/wheels
  wheels found: 60
installing from wheels (no network) ...
spec-2025.3.0
  Attempting uninstall: dask
    Found existing installation: dask 2026.1.1
    Uninstalling dask-2026.1.1:
      Successfully uninstalled dask-2026.1.1
  Attempting uninstall: click
    Found existing installation: click 8.3.3
    Uninstalling click-8.3.3:
      Successfully uninstalled click-8.3.3
  Attempting uninstall: charset-normalizer
    Found existing installation: charset-normalizer 3.4.7
    Uninstalling charset-normalizer-3.4.7:
      Successfully uninstalled charset-normalizer-3.4.7
  Attempting uninstall: certifi
    Found existing installation: certifi 2026.4.22
    Uninstalling certifi-2026.4.22:
      Successfully uninstalled certifi-2026.4.22
  Attempting uninstall: blosc2
    Found existing installation: blosc2 4.1.2
    Uninstalling blosc2-4.1.2:
      Successfully uninstalled blosc2-4.1.2

--- stderr tail ---

install 